# BPI2017 — Proceso de Préstamos (Loan Application)

**Dataset real** del BPI Challenge 2017 (~31k casos, proceso de solicitud de préstamos bancarios).  
Formato: **XES** (requiere `pm4py`).  
Referencia: [4TU Research Data](https://data.4tu.nl/articles/dataset/BPI_Challenge_2017/12696884)

### Particularidades respecto a SimBank
| Aspecto | SimBank | BPI2017 |
|---|---|---|
| Formato | pickle | XES / XES.gz |
| Columna case_id | `case_nr` | `case:concept:name` |
| Lifecycle filter | no | sí (keep 'complete') |
| Outcome | columna existente | engineered desde actividad `A_Accepted` |
| Intervención | `contact_headquarters` | `W_Call after offers` |
| Escala | 100k casos | 31k casos, ~200k eventos útiles |

### Dependencia
```bash
pip install pm4py
```

In [ ]:
from pathlib import Path

import xppm

print("xppm version:", xppm.__version__)

# Verificar que pm4py esté disponible (necesario para leer XES)
try:
    import pm4py

    print("pm4py version:", pm4py.__version__)
except ImportError:
    raise ImportError("Instala pm4py: pip install pm4py")

BASE = Path("..")
DATA = BASE / "data"
ART = BASE / "artifacts"
CFG = BASE / "configs" / "config.yaml"
DS = "bpi2017"

## 1. Definir schema para un log XES

Los logs XES producidos por `pm4py.convert_to_dataframe()` siempre usan las columnas estándar  
`case:concept:name`, `concept:name`, `time:timestamp`.  

BPI2017 registra múltiples transiciones por evento (start + complete).  
Activamos el **lifecycle filter** para quedarnos solo con los eventos `complete`.

In [ ]:
from xppm import EventLogSchema, LifecycleConfig, OutcomeConfig

schema = EventLogSchema(
    # Columnas estándar XES → pm4py
    case_id="case:concept:name",
    activity="concept:name",
    timestamp="time:timestamp",
    # XES tiene columnas extra (case:REG_DATE, case:AMOUNT_REQ, etc.)
    # Nos quedamos solo con las esenciales para reducir memoria
    select_cols=["case_id", "activity", "timestamp"],
    # Filtrar: quedarse solo con eventos 'complete'
    lifecycle=LifecycleConfig(
        enabled=True,
        column="lifecycle:transition",
        keep=["complete", "COMPLETE"],
    ),
    # Outcome: outcome=1 si el caso tiene la actividad A_Accepted en su traza
    outcome=OutcomeConfig(
        mode="from_activity",
        positive_activities=["A_Accepted"],
        output_col="outcome",
    ),
)

print("Schema:", schema.column_mapping)
print("Lifecycle filter habilitado:", schema.lifecycle.enabled)
print("Outcome desde actividades:", schema.outcome.positive_activities)

## 2. Preprocesar el log XES

Este paso puede tomar 1-2 minutos en la primera ejecución (la lectura XES.gz es el cuello de botella).

In [ ]:
from xppm import preprocess_event_log

raw_path = DATA / "raw" / "bpi2017.xes.gz"  # o "bpi2017.xes"
clean_path = DATA / DS / "interim" / "clean.parquet"
clean_path.parent.mkdir(parents=True, exist_ok=True)

stats = preprocess_event_log(raw_path, clean_path, schema=schema)

print(f"Casos tras filtrado:    {stats['n_cases']:>10,}")
print(f"Eventos útiles:         {stats['n_events']:>10,}")
print(f"Longitud media de traza: {stats['case_length_mean']:>9.1f} eventos/caso")
print()
# Verificar distribución del outcome
import pandas as pd  # noqa: E402

df = pd.read_parquet(clean_path)
outcome_by_case = df.groupby("case_id")["outcome"].first()
print(f"Casos aceptados (outcome=1): {outcome_by_case.sum():,} ({outcome_by_case.mean():.1%})")
print(f"Casos rechazados (outcome=0): {(1-outcome_by_case).sum():,}")

## 3. Exploración rápida del log preprocesado

In [ ]:
import pandas as pd

df = pd.read_parquet(clean_path)

print("Actividades más frecuentes:")
print(df["activity"].value_counts().head(10).to_string())

print("\n¿Cuántos casos tienen la intervención 'W_Call after offers'?")
n_interventions = (
    df.groupby("case_id")["activity"]
    .apply(lambda acts: (acts == "W_Call after offers").any())
    .sum()
)
total_cases = df["case_id"].nunique()
print(f"  {n_interventions:,} / {total_cases:,} ({n_interventions/total_cases:.1%})")

## 4–6. Codificar, construir MDP y particionar

Mismo flujo que SimBank, ahora con el config overlay de BPI2017.

In [ ]:
from xppm import Config, build_mdp_dataset, encode_prefixes, validate_and_split_dataset

cfg = Config.for_dataset(CFG, DS)  # carga config.yaml + configs/datasets/bpi2017.yaml

prefixes_path = DATA / DS / "interim" / "prefixes.npz"
vocab_path = DATA / DS / "interim" / "vocab_activity.json"
mdp_path = DATA / DS / "processed" / "D_offline.npz"
splits_path = DATA / DS / "processed" / "splits.json"

mdp_path.parent.mkdir(parents=True, exist_ok=True)
cfg.raw["encoding"]["output"]["vocab_activity_path"] = str(vocab_path)

# Codificar prefijos
enc_stats = encode_prefixes(clean_path, prefixes_path, config=cfg.raw)
print(f"Prefijos: {enc_stats['n_prefixes']:,}  |  Vocab: {enc_stats['vocab_size']} actividades")

# Construir MDP
mdp_stats = build_mdp_dataset(
    prefixes_path=prefixes_path,
    clean_log_path=clean_path,
    vocab_path=vocab_path,
    output_path=mdp_path,
    config=cfg.raw,
)
print(f"Transiciones MDP: {mdp_stats['n_transitions']:,}")

# Validar y partir
split_stats = validate_and_split_dataset(mdp_path, splits_path, config=cfg.raw)
for split, info in split_stats["splits"].items():
    print(f"  {split:5s}: {info['n_cases']:,} casos")

## 7. Entrenar TDQN

BPI2017 usa `max_steps=100_000` en producción. Aquí reducimos a `10_000` para demo.

In [ ]:
from xppm import TDQNConfig, train_tdqn

ckpt_dir = ART / "models" / "tdqn" / "bpi2017_demo"
ckpt_dir.mkdir(parents=True, exist_ok=True)

training_cfg = cfg.raw["training"]
transformer_cfg = training_cfg["transformer"]

tdqn_cfg = TDQNConfig(
    npz_path=str(mdp_path),
    splits_path=str(splits_path),
    vocab_path=str(vocab_path),
    # Arquitectura
    max_len=transformer_cfg.get("max_len", 50),
    d_model=transformer_cfg.get("d_model", 128),
    n_heads=transformer_cfg.get("n_heads", 4),
    n_layers=transformer_cfg.get("n_layers", 3),
    n_actions=len(cfg.raw["mdp"]["actions"]["id2name"]),  # 2: do_nothing | W_Call after offers
    # Hiperparámetros
    batch_size=training_cfg.get("batch_size", 128),
    learning_rate=training_cfg["tdqn"].get("learning_rate", 3e-4),
    gamma=training_cfg["tdqn"].get("gamma", 0.99),
    max_steps=10_000,  # demo; producción: 100_000
    eval_every=2_000,
    save_every=10_000,
    double_dqn=training_cfg["tdqn"].get("double_dqn", True),
    target_update_every=training_cfg["tdqn"].get("target_update_every", 2000),
    grad_clip_norm=training_cfg["tdqn"].get("grad_clip_norm", 10.0),
    device="cuda",
    seed=cfg.raw["repro"]["seed"],
)

result = train_tdqn(tdqn_cfg, checkpoint_dir=ckpt_dir)
ckpt_path = ckpt_dir / "Q_theta.ckpt"
print(f"Checkpoint: {ckpt_path}")
print(f"Loss final: {result['final_loss']:.4f}   Q-mean: {result['final_q_mean']:.4f}")

## 8. Evaluar política (OPE)

In [ ]:
from xppm import doubly_robust_estimate, fit_behavior_policy_tdqn_encoder

behavior = fit_behavior_policy_tdqn_encoder(
    npz_path=mdp_path,
    splits_path=splits_path,
    ckpt_path=ckpt_path,
    vocab_path=vocab_path,
    config=cfg.raw,
)

ope = doubly_robust_estimate(
    ckpt_path=ckpt_path,
    dataset_path=mdp_path,
    splits_path=splits_path,
    vocab_path=vocab_path,
    config=cfg.raw,
    behavior=behavior,
)

res = ope["results"]
print(
    f"π_e  (TDQN):      {res['pi_e']['dr_mean']:.4f}  "
    f"CI [{res['pi_e']['ci_low']:.4f}, {res['pi_e']['ci_high']:.4f}]"
)
print(f"μ (histórica):    {res['behavior']['dr_mean']:.4f}")
uplift = res["pi_e"]["dr_mean"] - res["behavior"]["dr_mean"]
print(f"Δ (uplift TDQN):  {uplift:+.4f}")

## 9. Explicaciones de política

Usamos el checkpoint entrenado para calcular atribuciones con Integrated Gradients.

In [ ]:
import json

from xppm import explain_policy

cfg.raw["xai"]["checkpoint_path"] = str(ckpt_path)
cfg.raw["xai"]["n_cases"] = 50
cfg.raw["xai"]["out_dir"] = f"xai/{DS}"

xai_paths = explain_policy(cfg.raw, config_hash=cfg.config_hash)

# Resumen de top actividades globales (por frecuencia de aparición en top-k)
risk_data = json.loads(xai_paths["risk"].read_text())

print(f"Top actividades que explican riesgo ({len(risk_data['items'])} transiciones):")
top_global = risk_data["metadata"].get("top_tokens_risk", [])
for entry in top_global[:8]:
    name = entry.get("token_name", f"token_{entry['token_id']}")
    print(
        f"  {name:<40s}  frecuencia={entry['frequency']:3d}  "
        f"importancia_mediana={entry['median_importance']:.4f}"
    )

## 10. Visualización: importancia de actividades por paso de proceso

Muestra qué actividades son más importantes en función de la longitud del prefijo.

In [ ]:
import json

risk_data = json.loads(xai_paths["risk"].read_text())
items = risk_data["items"]

# Importancia media de cada actividad por longitud de prefijo (t)
from collections import defaultdict  # noqa: E402

token_importance_by_step: dict = defaultdict(list)
for item in items:
    t = item["t"]
    for tok in item["top_tokens"]:
        key = (tok["token_name"], t)
        token_importance_by_step[key].append(tok["importance"])

# Top 5 actividades más frecuentes en las explicaciones
activity_counts: dict = defaultdict(int)
for item in items:
    for tok in item["top_tokens"]:
        activity_counts[tok["token_name"]] += 1

top5 = sorted(activity_counts.items(), key=lambda x: -x[1])[:5]
print("Top 5 actividades más frecuentes en explicaciones de riesgo:")
for name, cnt in top5:
    print(f"  {name:<40s}: aparece en {cnt} transiciones")